# Artifact Manager Tutorial

This notebook walks through the **`artifact_manager`** module - how to save and load experiment results.

### What it does
- Stores results as **`meta.json` + `data.npz`** inside named directories
- Lets you pick **human-readable run names** (e.g. `heisenberg_N10_pt`)
- Keeps the **config hash inside `meta.json`** for integrity checking
- Validates column lengths on save
- Provides **discovery**: find runs by name, hash prefix, or meta-field queries

### On-disk layout
```
results_root/
└── <artifact_name>/
    └── v<version>/
        └── <run_name>/          ← you choose this name!
            ├── meta.json        ← config, hash, timestamps, provenance
            └── data.npz         ← result arrays
```

## 1. Setup

In [ ]:
import sys, json, shutil
import numpy as np
from pathlib import Path
import os

# Add the project root to sys.path so top-level modules are importable
PROJECT_ROOT = str(Path("..").resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from artifact_manager import ArtifactManager, RunDir, config_hash

demo_root = Path("artifact_demo")
if demo_root.exists():
    shutil.rmtree(demo_root)   # start fresh on each run
demo_root.mkdir()

am_demo = ArtifactManager(demo_root)
print(f"Demo results root: {demo_root}")

Demo results root: artifact_demo


## 2. The API at a glance

The module has two classes:

- **`ArtifactManager`** — manages the results root, creates/finds/lists runs
- **`RunDir`** — handle to one run directory, with methods for saving and loading data

Here are their public methods:

In [4]:
# The public API is intentionally small:
import inspect

for name, obj in inspect.getmembers(ArtifactManager):
    if not name.startswith("_") and callable(obj):
        sig = inspect.signature(obj)
        print(f"  ArtifactManager.{name}{sig}")

print()
for name, obj in inspect.getmembers(RunDir):
    if not name.startswith("_") and callable(obj):
        sig = inspect.signature(obj)
        print(f"  RunDir.{name}{sig}")

  ArtifactManager.config_hash(config: 'Mapping[str, Any]', *, n_chars: 'int' = 16) -> 'str'
  ArtifactManager.create_run(self, *, artifact: 'str', name: 'str', config: 'Mapping[str, Any]', version: 'int' = 1) -> 'RunDir'
  ArtifactManager.find(self, *, artifact: 'str', version: 'int' = 1, name: 'str | None' = None, config_hash_prefix: 'str | None' = None, meta_query: 'Mapping[str, Any] | None' = None) -> 'RunDir'
  ArtifactManager.list_runs(self, artifact: 'str', version: 'int' = 1) -> 'List[RunDir]'
  ArtifactManager.open_run(self, *, artifact: 'str', name: 'str', version: 'int' = 1) -> 'RunDir'

  RunDir.load_meta(self) -> 'Dict[str, Any]'
  RunDir.load_records(self, *, key: 'str', fields: 'Mapping[str, np.dtype]') -> 'Dict[int, Dict[str, Any]]'
  RunDir.load_table(self) -> 'Dict[str, np.ndarray]'
  RunDir.save_meta(self, meta: 'MutableMapping[str, Any]') -> 'None'
  RunDir.save_records(self, *, key: 'str', fields: 'Mapping[str, np.dtype]', records: 'Mapping[int, Mapping[str, Any]]')

## 3. How naming works

You choose a **human-readable name** for each run. The directory structure looks like:

```
results_root/spin_optimization_sweep/v1/
├── heisenberg_N10_periodic/
├── ising_N8_open/
└── xxz_N12_sweep/
```

A deterministic **config hash** is computed from your config dict and stored inside `meta.json` - this is used for deduplication and lookup.

In [5]:
# The config hash is deterministic — same config always gives the same hash
sample_config = {"model": "heisenberg", "boundary": "periodic", "N": 10}
h = config_hash(sample_config)
print(f"Config hash: {h}")

Config hash: 025f72e922669e3d


## 4. Creating a run

The core workflow is:

1. **Create** an `ArtifactManager` pointing at your results root.
2. **Create a run** with `am.create_run(artifact, name, config)`.
3. **Save data** with `run.save_records(...)` or `run.save_table(...)`.

The `create_run` call:
- Creates the directory `<results_root>/<artifact>/v1/<name>/`
- Writes `meta.json` with your config **plus** the config hash, timestamps, and provenance info

In [6]:
config = {"model": "heisenberg_simple", "boundary": "periodic", "N": 10}

run = am_demo.create_run(
    artifact="spin_optimization_sweep",
    name="heisenberg_N5_periodic",      # ← human-readable!
    config=config,
)

print(f"Run directory: {run.path}")
print(f"Run name:      {run.path.name}")

# Let's inspect the meta.json that was auto-created
meta = run.load_meta()
print("\nmeta.json contents:")
print(json.dumps(meta, indent=2))

Run directory: artifact_demo/spin_optimization_sweep/v1/heisenberg_N5_periodic
Run name:      heisenberg_N5_periodic

meta.json contents:
{
  "N": 10,
  "boundary": "periodic",
  "config_hash": "ec75d9a38014f7c1",
  "created_at": "2026-02-17T10:26:33Z",
  "model": "heisenberg_simple",
  "platform": {
    "machine": "x86_64",
    "release": "5.14.0-611.30.1.el9_7.x86_64",
    "system": "Linux"
  },
  "python": "3.13.11 | packaged by Anaconda, Inc. | (main, Dec 10 2025, 21:28:48) [GCC 14.3.0]",
  "updated_at": "2026-02-17T10:26:33Z"
}


Notice the `config_hash` field inside `meta.json` — this lets you look up runs by hash prefix even when the folder has a descriptive name. The `name` parameter is **required**, so every run gets a clear, meaningful directory name.

## 5. Saving & loading **keyed records** (one row per unique key)

Use this pattern when each row is identified by a unique key (e.g. system size `N`).

**Scripts that use this pattern:** `exact_ground_energy.py`, `npa_energy_lb.py`, `dmrg.py`

The API:
```python
fields = {"N": np.int64, "energy": np.float64, "time_s": np.float64}
records = {4: {"N": 4, "energy": -1.616, "time_s": 0.01},
           6: {"N": 6, "energy": -2.494, "time_s": 0.05}}
run.save_records(key="N", fields=fields, records=records)
loaded = run.load_records(key="N", fields=fields)  # → same dict structure
```

`save_records` is **upsert-style**: call it again with updated records and it overwrites. The data is stored as sorted column arrays in `data.npz`.

In [7]:
# Create a run for exact ground energies
run_exact = am_demo.create_run(
    artifact="spin_exact_ground_energy",
    name="heisenberg_chain periodic",
    config={"model": "heisenberg", "boundary": "periodic"},
)

# Define the schema: field name → dtype
fields = {"N": np.int64, "energy": np.float64, "time_s": np.float64}

# Build records dict: key (N) → {field: value}
records = {
    4:  {"N": 4,  "energy": -1.616, "time_s": 0.01},
    6:  {"N": 6,  "energy": -2.494, "time_s": 0.05},
    8:  {"N": 8,  "energy": -3.375, "time_s": 0.30},
}

run_exact.save_records(key="N", fields=fields, records=records)
print(f"Saved {len(records)} records")

# Load everything back
loaded = run_exact.load_records(key="N", fields=fields)
print(f"\nLoaded {len(loaded)} records:")
for k in sorted(loaded):
    r = loaded[k]
    print(f"  N={r['N']}, energy={r['energy']:.3f}, time={r['time_s']:.2f}s")

# Now add N=10 — merge with existing and re-save
loaded[10] = {"N": 10, "energy": -4.258, "time_s": 1.2}
run_exact.save_records(key="N", fields=fields, records=loaded)
reloaded = run_exact.load_records(key="N", fields=fields)
print(f"\nAfter adding N=10: {len(reloaded)} records total")

Saved 3 records

Loaded 3 records:
  N=4, energy=-1.616, time=0.01s
  N=6, energy=-2.494, time=0.05s
  N=8, energy=-3.375, time=0.30s

After adding N=10: 4 records total


## 6. Saving & loading **flat tables** (multiple rows per key)

Use this pattern when you have many rows (e.g. multiple seeds per system size).

**Scripts that use this pattern:** `optimization_sweep.py`, `fraction_scaling.py`, `sdp_benchmark.py`, `symmetry_benchmark.py`

The API:
```python
run.save_table(k=k_arr, seed=seed_arr, energy=energy_arr)
data = run.load_table()  # returns dict of arrays
```

In [8]:
# Create a run for an optimization sweep
run_sweep = am_demo.create_run(
    artifact="spin_optimization_sweep",
    name="heisenberg_sweep_k2to5_p05",
    config={"model": "heisenberg", "boundary": "periodic", "N": 9, "p": 0.5},
)

# Simulate results: 3 seeds × 4 k-values = 12 rows
k_vals = np.repeat([2, 3, 4, 5], 3)
seeds  = np.tile([42, 43, 44], 4)
energy = -3.0 + np.random.randn(12) * 0.1
time_s = np.abs(np.random.randn(12)) * 0.5

run_sweep.save_table(k=k_vals, seed=seeds, energy_lb=energy, time_s=time_s)
print(f"Saved table with {len(k_vals)} rows")

# Load it back
data = run_sweep.load_table()
print(f"\nLoaded columns: {sorted(data.keys())}")
print(f"Number of rows: {len(data['k'])}")
print(f"\nFirst 5 rows:")
for i in range(5):
    print(f"  k={data['k'][i]}, seed={data['seed'][i]}, "
          f"energy_lb={data['energy_lb'][i]:.3f}, time={data['time_s'][i]:.2f}s")

Saved table with 12 rows

Loaded columns: ['energy_lb', 'k', 'seed', 'time_s']
Number of rows: 12

First 5 rows:
  k=2, seed=42, energy_lb=-2.979, time=0.61s
  k=2, seed=43, energy_lb=-3.039, time=0.88s
  k=2, seed=44, energy_lb=-2.951, time=0.27s
  k=3, seed=42, energy_lb=-2.888, time=0.10s
  k=3, seed=43, energy_lb=-2.877, time=0.15s


### Column-length validation

`save_table` **rejects** mismatched array lengths:

In [9]:
# This will raise a ValueError!
try:
    run_sweep.save_table(
        k=np.array([1, 2, 3]),
        energy=np.array([0.1, 0.2]),  # ← length 2 vs length 3!
    )
except ValueError as e:
    print(f"✅ Caught expected error:\n   {e}")

✅ Caught expected error:
   All arrays must have the same first-axis length.  Got: energy=2, k=3


## 7. Discovery — finding runs by name, hash, or config query

You can **search** for runs without remembering exact folder paths.

| Method | Use case |
|--------|----------|
| `am.find(artifact, name="...")` | Find by exact folder name |
| `am.find(artifact, config_hash_prefix="abc...")` | Find by (partial) config hash |
| `am.find(artifact, meta_query={...})` | Find all runs whose `meta.json` contains matching fields |
| `am.list_runs(artifact)` | List all runs for an artifact |

In [10]:
# List all runs for an artifact
print("All 'spin_optimization_sweep' runs:")
for r in am_demo.list_runs("spin_optimization_sweep"):
    print(f"  {r.path.name}")

print()

# Find by exact name
found = am_demo.find(artifact="spin_optimization_sweep", name="heisenberg_sweep_k2to5_p05")
print(f"Find by name: {found}")

# Find by config hash prefix (first 6 chars is usually enough)
h = config_hash({"model": "heisenberg", "boundary": "periodic", "N": 9, "p": 0.5})
found_by_hash = am_demo.find(artifact="spin_optimization_sweep", config_hash_prefix=h[:6])
print(f"Find by hash prefix '{h[:6]}...': {found_by_hash}")

# Find by meta query (subset matching)
# If multiple candidates match, the most recently updated run wins
found_by_query = am_demo.find(artifact="spin_optimization_sweep", meta_query={"N": 9})
print(f"Find by meta_query {{N: 9}}: {found_by_query}")

All 'spin_optimization_sweep' runs:
  heisenberg_N5_periodic
  heisenberg_sweep_k2to5_p05

Find by name: RunDir('artifact_demo/spin_optimization_sweep/v1/heisenberg_sweep_k2to5_p05')
Find by hash prefix '1222fe...': RunDir('artifact_demo/spin_optimization_sweep/v1/heisenberg_sweep_k2to5_p05')
Find by meta_query {N: 9}: RunDir('artifact_demo/spin_optimization_sweep/v1/heisenberg_sweep_k2to5_p05')


## 8. Reading existing results

Point an `ArtifactManager` at any results directory to browse and load data:

In [11]:
am_demo = ArtifactManager(Path("artifact_demo"))

# List all artifact types
print("=== Available artifacts ===")
for artifact_dir in sorted(am_demo.root.iterdir()):
    if artifact_dir.is_dir():
        runs = am_demo.list_runs(artifact_dir.name)
        print(f"  {artifact_dir.name}: {len(runs)} run(s)")

# List runs and their configs
print("\n=== spin_exact_ground_energy runs ===")
for r in am_demo.list_runs("spin_exact_ground_energy"):
    meta = r.load_meta()
    model = meta.get("model", "?")
    boundary = meta.get("boundary", "?")
    print(f"  {r.path.name}  →  model={model}, boundary={boundary}")

# Load data from one of them
runs = am_demo.list_runs("spin_exact_ground_energy")
if runs:
    r = runs[0]
    data = r.load_table()
    print(f"\n=== Data from {r.path.name} ===")
    print(f"  Columns: {sorted(data.keys())}")
    for col, arr in sorted(data.items()):
        print(f"  {col}: shape={arr.shape}, dtype={arr.dtype}")

=== Available artifacts ===
  spin_exact_ground_energy: 1 run(s)
  spin_optimization_sweep: 2 run(s)

=== spin_exact_ground_energy runs ===
  heisenberg_chain periodic  →  model=heisenberg, boundary=periodic

=== Data from heisenberg_chain periodic ===
  Columns: ['N', 'energy', 'time_s']
  N: shape=(4,), dtype=int64
  energy: shape=(4,), dtype=float64
  time_s: shape=(4,), dtype=float64


## 9. Typical script pattern

Every write in `artifact_manager` is **atomic** — it writes to a temp file then does an `os.replace`, so a crash or `Ctrl-C` can never leave a half-written `data.npz` or `meta.json` on disk.

Here's the standard pattern for using `artifact_manager` in an experiment script:

```python
from spins_sdp.scripts.artifact_manager import ArtifactManager

FIELDS = {"N": np.int64, "energy": np.float64, "time_s": np.float64}

am = ArtifactManager("spins_sdp/results")

# Create or reopen a run
run = am.create_run(
    artifact="spin_exact_ground_energy",
    name="heisenberg_periodic",
    config={"model": "heisenberg", "boundary": "periodic"},
)

# Resume: load existing data if present
existing = run.load_records(key="N", fields=FIELDS)

for N in [4, 6, 8, 10, 12]:
    if N in existing:
        continue  # already done — skip

    energy, dt = compute(N)
    existing[N] = {"N": N, "energy": energy, "time_s": dt}

    # Atomic save after EACH new result
    #    - a crash here loses at most the current N, not previous ones
    run.save_records(key="N", fields=FIELDS, records=existing)
    run.update_meta(Ns_present=sorted(existing.keys()))

run.update_meta(status="complete")
```

## 10. Updating metadata after a run

You can add extra fields to `meta.json` at any time with `update_meta()`. This is useful for recording results summaries, notes, or marking runs as complete:

In [12]:
# Re-open an existing run
run = am_demo.open_run(artifact="spin_exact_ground_energy", name="heisenberg_chain periodic")

# Add some metadata after the fact
run.update_meta(
    status="complete",
    Ns_present=[4, 6, 8, 10],
    best_energy=-4.258,
    notes="Converged for all system sizes"
)

# Verify
meta = run.load_meta()
print("Updated meta.json:")
print(json.dumps(meta, indent=2))

Updated meta.json:
{
  "Ns_present": [
    4,
    6,
    8,
    10
  ],
  "best_energy": -4.258,
  "boundary": "periodic",
  "config_hash": "d34187c39f23cc5e",
  "created_at": "2026-02-17T10:26:37Z",
  "model": "heisenberg",
  "notes": "Converged for all system sizes",
  "platform": {
    "machine": "x86_64",
    "release": "5.14.0-611.30.1.el9_7.x86_64",
    "system": "Linux"
  },
  "python": "3.13.11 | packaged by Anaconda, Inc. | (main, Dec 10 2025, 21:28:48) [GCC 14.3.0]",
  "status": "complete",
  "updated_at": "2026-02-17T10:26:54Z"
}


---

## Quick Reference

| Task | Code |
|------|------|
| Create manager | `am = ArtifactManager(Path("spins_sdp/results"))` |
| Create a run | `run = am.create_run(artifact="name", name="my_run", config={...})` |
| Open existing run | `run = am.open_run(artifact="name", name="my_run")` |
| Save keyed records | `run.save_records(key="N", fields={"N": np.int64, ...}, records={4: {...}})` |
| Load keyed records | `data = run.load_records(key="N", fields={"N": np.int64, ...})` |
| Save flat table | `run.save_table(col1=arr1, col2=arr2, ...)` |
| Load flat table | `data = run.load_table()` |
| Update metadata | `run.update_meta(key="value")` |
| Find by name | `am.find(artifact="name", name="my_run")` |
| Find by hash | `am.find(artifact="name", config_hash_prefix="abc123")` |
| Find by query | `am.find(artifact="name", meta_query={"N": 9})` |
| List all runs | `am.list_runs("artifact_name")` |


### On-disk layout
```
results/
  spin_exact_ground_energy/
    v1/
      heisenberg_periodic/
        meta.json               ← config + config_hash + provenance
        data.npz                ← numpy arrays
      ising_periodic/
        meta.json
        data.npz
```